# A Systemic Risk Framework for the Vietnamese Market: a Composite Risk Index and Financial Stress Prediction

## 1. Collect the Data

Import the needed libraries.

In [1]:
import requests
import pandas as pd
from functools import reduce
import io
import time
import yfinance as yf
try:
  import akshare as ak
except:
  !pip install akshare
  import akshare as ak
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import drive
from google.colab import files
import os

Build a Data Information Table for the Vietnamese equity data, including:
- Ticker;
- Company Name;
- Sector: the industry the company operates in;
- Risk Transmission Channel: a macro-economic sector that represents a specific source or amplifier of shocks, spreading systemic risk within and across sectors.

In [2]:
ticker_info = {
    "NT2": {"Risk Transmission Channel": "Energy & Infrastructure", "Sector": "Energy", "Company": "PetroVietnam Power Nhon Trach 2 JSC"},
    "PVD": {"Risk Transmission Channel": "Energy & Infrastructure", "Sector": "Energy", "Company": "PetroVietnam Drilling & Well Services Corp"},
    "PVS": {"Risk Transmission Channel": "Energy & Infrastructure", "Sector": "Energy", "Company": "PetroVietnam Technical Services Corp"},
    "PVT": {"Risk Transmission Channel": "Energy & Infrastructure", "Sector": "Energy", "Company": "PetroVietnam Transportation Corp"},
    "REE": {"Risk Transmission Channel": "Energy & Infrastructure", "Sector": "Utilities / Infrastructure", "Company": "Refrigeration Electrical Engineering Corp"},
    "ACB": {"Risk Transmission Channel": "Financial System", "Sector": "Finance - Banking", "Company": "Asia Commercial Joint Stock Bank"},
    "CTG": {"Risk Transmission Channel": "Financial System", "Sector": "Finance - Banking", "Company": "VietinBank"},
    "MBB": {"Risk Transmission Channel": "Financial System", "Sector": "Finance - Banking", "Company": "Military Commercial JSB"},
    "SHB": {"Risk Transmission Channel": "Financial System", "Sector": "Finance - Banking", "Company": "Saigon-Hanoi Commercial JSB"},
    "STB": {"Risk Transmission Channel": "Financial System", "Sector": "Finance - Banking", "Company": "Saigon Thuong Tin Commercial JSB"},
    "VCB": {"Risk Transmission Channel": "Financial System", "Sector": "Finance - Banking", "Company": "Vietcombank"},
    "BVH": {"Risk Transmission Channel": "Financial System", "Sector": "Finance - Insurance", "Company": "Bao Viet Holdings"},
    "PVI": {"Risk Transmission Channel": "Financial System", "Sector": "Finance - Insurance", "Company": "PVI Holdings"},
    "SSI": {"Risk Transmission Channel": "Financial System", "Sector": "Finance - Securities/Brokerage", "Company": "SSI Securities Corp"},
    "VIX": {"Risk Transmission Channel": "Financial System", "Sector": "Finance - Securities/Brokerage", "Company": "VIX Securities JSC"},
    "VND": {"Risk Transmission Channel": "Financial System", "Sector": "Finance - Securities/Brokerage", "Company": "VNDirect Securities Corp"},
    "GMD": {"Risk Transmission Channel": "Logistics", "Sector": "Logistics", "Company": "Gemadept Corp"},
    "VSC": {"Risk Transmission Channel": "Logistics", "Sector": "Logistics", "Company": "Vietnam Container Shipping JSC"},
    "MSN": {"Risk Transmission Channel": "Real Economy", "Sector": "Consumer Goods / Retail", "Company": "Masan Group Corp"},
    "PNJ": {"Risk Transmission Channel": "Real Economy", "Sector": "Consumer Goods / Retail", "Company": "Phu Nhuan Jewelry JSC"},
    "VNM": {"Risk Transmission Channel": "Real Economy", "Sector": "Consumer Goods / Retail", "Company": "Vietnam Dairy Products JSC"},
    "CSM": {"Risk Transmission Channel": "Real Economy", "Sector": "Industrial / Materials", "Company": "Casumina"},
    "DPM": {"Risk Transmission Channel": "Real Economy", "Sector": "Industrial / Materials", "Company": "PetroVietnam Fertilizer and Chemicals Corp"},
    "HPG": {"Risk Transmission Channel": "Real Economy", "Sector": "Materials / Industrial", "Company": "Hoa Phat Group JSC"},
    "HSG": {"Risk Transmission Channel": "Real Economy", "Sector": "Materials / Industrial", "Company": "Hoa Sen Group"},
    "NKG": {"Risk Transmission Channel": "Real Economy", "Sector": "Materials / Industrial", "Company": "Nam Kim Steel JSC"},
    "FPT": {"Risk Transmission Channel": "Real Economy", "Sector": "Technology", "Company": "FPT Corp"},
    "DIG": {"Risk Transmission Channel": "Real Estate", "Sector": "Real Estate", "Company": "Development Investment Construction JSC"},
    "DXG": {"Risk Transmission Channel": "Real Estate", "Sector": "Real Estate", "Company": "Dat Xanh Group JSC"},
    "KBC": {"Risk Transmission Channel": "Real Estate", "Sector": "Real Estate", "Company": "Kinh Bac City Development Holding Corp"},
    "KDH": {"Risk Transmission Channel": "Real Estate", "Sector": "Real Estate", "Company": "Khang Dien House Trading and Investment JSC"},
    "VIC": {"Risk Transmission Channel": "Real Estate", "Sector": "Real Estate", "Company": "Vingroup JSC"},
}
ticker_df = (
    pd.DataFrame.from_dict(ticker_info, orient="index")
      .reset_index()
      .rename(columns={"index": "ticker"})
      [["Risk Transmission Channel", "Sector", "Company", "ticker"]]
)

ticker_df

,Risk Transmission Channel,Sector,Company,ticker
0,Energy & Infrastructure,Energy,PetroVietnam Power Nhon Trach 2 JSC,NT2
1,Energy & Infrastructure,Energy,PetroVietnam Drilling & Well Services Corp,PVD
2,Energy & Infrastructure,Energy,PetroVietnam Technical Services Corp,PVS
3,Energy & Infrastructure,Energy,PetroVietnam Transportation Corp,PVT
4,Energy & Infrastructure,Utilities / Infrastructure,Refrigeration Electrical Engineering Corp,REE
5,Financial System,Finance - Banking,Asia Commercial Joint Stock Bank,ACB
6,Financial System,Finance - Banking,VietinBank,CTG
7,Financial System,Finance - Banking,Military Commercial JSB,MBB
8,Financial System,Finance - Banking,Saigon-Hanoi Commercial JSB,SHB
9,Financial System,Finance - Banking,Saigon Thuong Tin Commercial JSB,STB


Build a function to collect Vietnamese stock data from <a href='https://cafef.vn/'>CafeF</a>.

In [3]:
session = requests.Session()
headers = {"User-Agent": "Mozilla/5.0", "Referer": "https://cafef.vn/", "Origin": "https://cafef.vn", "X-Requested-With": "XMLHttpRequest",}

def fetch_price_history_cafef(ticker, start="2012-01-01", end="2026-03-31"):
  url = "https://cafef.vn/du-lieu/Ajax/PageNew/DataHistory/PriceHistory.ashx"

  all_data = []
  page = 1

  while True:
    params = {"Symbol":ticker,"StartDate":"","EndDate":"","PageIndex":page,"PageSize":100}

    try:
      r = session.get(url, headers=headers, params=params, timeout=15)
      r.raise_for_status()
      json_data = r.json()
    except Exception as e:
      print(f"{ticker}: request failed -> {e}")
      break

    if not json_data or json_data.get("Data") is None:
      print(f"{ticker}: no data returned (page {page})")
      break

    data = json_data["Data"].get("Data", [])

    if not data:
      break

    all_data.extend(data)

    # stop if last page
    if len(data) < 100:
      break

    page += 1
    time.sleep(0.1)  # avoid throttling

  if not all_data:
    print(f"{ticker}: EMPTY")
    return pd.DataFrame(columns=["Date", f"{ticker}_close", f"{ticker}_volume"])

  df = pd.DataFrame(all_data).rename(columns={"Ngay": "Date","GiaDieuChinh": f"{ticker}_close","KhoiLuongKhopLenh": f"{ticker}_volume"})

  df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
  df[f"{ticker}_close"] = pd.to_numeric(df[f"{ticker}_close"], errors="coerce")
  df[f"{ticker}_volume"] = pd.to_numeric(df[f"{ticker}_volume"], errors="coerce")
  start_dt = pd.to_datetime(start, format="%Y-%m-%d")
  end_dt = pd.to_datetime(end, format="%Y-%m-%d")
  df = df[(df["Date"] >= start_dt) & (df["Date"] <= end_dt)]

  return df[["Date", f"{ticker}_close", f"{ticker}_volume"]]

Collect the Vietnamese equity data from <a href='https://cafef.vn/'>CafeF.</a>

```python
def fetch_wrapper(t):
    print(f"Fetching {t}...")
    df = fetch_price_history_cafef(t)
    return t, df

dfs = []

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(fetch_wrapper, t) for t in ticker_info.keys()]

    for future in as_completed(futures):
        t, df = future.result()
        if not df.empty:
            dfs.append(df)
        else:
            print(f"{t}: dropped")
```
Fetching NT2...  
Fetching PVD...  
Fetching PVS...  
Fetching PVT...  
Fetching REE...  
Fetching ACB...  
Fetching CTG...  
Fetching MBB...  
Fetching SHB...  
Fetching STB...  
Fetching VCB...  
Fetching BVH...  
Fetching PVI...  
Fetching SSI...  
Fetching VIX...  
Fetching VND...  
Fetching GMD...  
Fetching VSC...  
Fetching MSN...  
Fetching PNJ...  
Fetching VNM...  
Fetching CSM...  
Fetching DPM...  
Fetching HPG...  
Fetching HSG...  
Fetching NKG...  
Fetching FPT...  
Fetching DIG...  
Fetching DXG...  
Fetching KBC...  
Fetching KDH...  
Fetching VIC...  

Save the dataframe to avoid fetching the data from <a href='https://cafef.vn/'>CafeF</a> again.

```python
df_vn = reduce(lambda left, right: pd.merge(left, right, on="Date", how="outer"),dfs)
df_vn = df_vn.sort_values("Date").reset_index(drop=True)
df_vn.to_csv("df_vn.csv", index=False)  # saves in VM
files.download("df_vn.csv")
df_vn.to_parquet("df_vn.parquet", index=False)
files.download("df_vn.parquet")
```

Import the dataframe to the notebook.

In [4]:
file_path = "/content/drive/My Drive/WQU/10_Capstone_Project/Capstone Project/data/df_vn.parquet"
def import_data(file_path):
  try:
    drive.mount('/content/drive', force_remount=True)
    # Check if file exists
    if os.path.exists(file_path):
      df = pd.read_parquet(file_path)
      print(f"Loaded dataframe from Drive ({file_path})")
    else:
      raise FileNotFoundError(f"File not found at {file_path}")

  except Exception as e:
    print(f"Drive not available or file missing: {e}")
    print("Please upload dataframe manually.")
    uploaded = files.upload()

    # Automatically read the uploaded file
    file_name = list(uploaded.keys())[0]  # pick the first uploaded file
    df = pd.read_parquet(io.BytesIO(uploaded[file_name]))
    print(f"Loaded {file_name} from manual upload")
    return df

In [5]:
df_vn = import_data(file_path)

Drive not available or file missing: Error: credential propagation was unsuccessful
Please upload dataframe manually.


Saving df_vn.parquet to df_vn (5).parquet
Loaded df_vn (5).parquet from manual upload


In [6]:
df_vn.head()

,Date,NT2_close,NT2_volume,PVT_close,PVT_volume,PVS_close,PVS_volume,PVD_close,PVD_volume,REE_close,...,DIG_close,DIG_volume,DXG_close,DXG_volume,KBC_close,KBC_volume,KDH_close,KDH_volume,VIC_close,VIC_volume
0,2012-01-03,1.06,16500.0,1.01,33120.0,3.73,77900.0,13.42,40680.0,2.77,...,2.04,105040.0,1.17,30690.0,4.96,315520.0,3.60,55300.0,7.81,39190.0
1,2012-01-04,1.06,12000.0,1.04,457630.0,3.50,285700.0,12.79,97220.0,2.85,...,2.00,142770.0,1.15,28520.0,5.01,70540.0,3.73,61980.0,7.85,38760.0
2,2012-01-05,0.97,0.0,1.07,621160.0,3.42,446500.0,12.74,66780.0,2.85,...,1.96,66330.0,1.13,33310.0,4.91,43340.0,3.90,52010.0,7.49,15860.0
3,2012-01-06,1.06,2900.0,1.04,282320.0,3.34,94100.0,12.74,15050.0,2.80,...,1.91,242390.0,1.12,13890.0,4.91,29410.0,3.86,37940.0,7.45,22080.0
4,2012-01-09,1.10,1100.0,1.04,231330.0,3.29,125400.0,12.58,31050.0,2.90,...,1.85,630070.0,1.10,6770.0,4.82,59880.0,3.81,53980.0,7.81,14940.0


In [7]:
df_vn.tail()

,Date,NT2_close,NT2_volume,PVT_close,PVT_volume,PVS_close,PVS_volume,PVD_close,PVD_volume,REE_close,...,DIG_close,DIG_volume,DXG_close,DXG_volume,KBC_close,KBC_volume,KDH_close,KDH_volume,VIC_close,VIC_volume
3547,2026-03-25,27.70,2979600.0,21.70,8810800.0,42.2,7282100.0,34.75,3336600.0,70.9,...,13.80,11696900.0,14.15,20550300.0,28.9,4274000.0,25.85,5073300.0,128.7,3013400.0
3548,2026-03-26,27.75,1455700.0,22.40,13267800.0,42.0,5057500.0,34.70,4219300.0,71.7,...,13.55,8624500.0,13.85,15497700.0,28.7,1780000.0,25.45,2692100.0,130.0,2435600.0
3549,2026-03-27,27.90,950600.0,22.30,9317400.0,42.8,5938400.0,35.75,5186400.0,71.7,...,14.45,24713800.0,14.60,41747300.0,30.7,6091600.0,26.40,4704200.0,132.6,2124700.0
3550,2026-03-30,27.90,1371700.0,21.85,12201400.0,42.8,5262900.0,36.20,4041300.0,70.0,...,14.20,13413500.0,14.30,16998400.0,31.8,7130700.0,26.05,4370900.0,129.5,2042500.0
3551,2026-03-31,27.60,846800.0,21.80,7238400.0,40.9,7568200.0,34.80,5738900.0,68.5,...,14.25,11416400.0,14.55,21018600.0,31.9,2270100.0,26.00,4193600.0,135.0,3316700.0


In [8]:
df_vn.shape

(3552, 65)

___

Gather VNIndex historical data (source: <a href='https://cafef.vn/'>CafeF</a>).

```python
df_vnindex = fetch_price_history_cafef("VNINDEX", start="2012-01-01", end="2026-03-31")
df_vnindex = df_vnindex.sort_values("Date").reset_index(drop=True)
```

Collect data for external financial markets: advanced and regionally dominant economies (sources: <a href='https://finance.yahoo.com/'>Yahoo Finance</a> and <a href='https://github.com/akfamily/akshare'>AKShare</a>).

```python
foreign_index = {
    "^GSPC": "S&P 500 (US)",
    "^GSPTSE": "S&P/TSX (Canada)",
    "^STOXX50E": "EURO STOXX 50 (EU)",
    "^FTSE": "FTSE 100 (UK)",
    # "000300.SS": "CSI 300 (China)",
    "^N225": "Nikkei 225 (Japan)",
    "^KS11": "KOSPI (S.Korea)",
    "^AXJO": "S&P/ASX 200 (Australia)",
    "^HSI": "Hang Seng (HK)",
    "^TWII": "TWSE (Taiwan)"
}
foreign_index.keys()
df_fx = yf.download(list(foreign_index.keys()), start="2012-01-01", end="2026-04-01", auto_adjust= True)[['Close']]#, 'Volume']]

df_fx.columns =  [f"{ticker}_{level}" for level, ticker in df_fx.columns]
```

```python
df_CSI300= (
    ak.stock_zh_index_daily(symbol="sh000300")
      .assign(date=lambda x: pd.to_datetime(x["date"]))
      .set_index("date")
      .loc["2012-01-01":"2026-03-31", ["close"]]#, "volume"]]
      .rename_axis(None)
      .rename(columns={"close": "CSI300_close"})#, "volume": "CSI300_volume"})
      .reset_index()
      .rename(columns={"index": "Date", "date": "Date"})
      .set_index("Date")
)
```

```python
# combine all international data
df_glob = (df_vnindex.merge(df_fx, on="Date", how="outer").merge(df_CSI300, on="Date", how="outer").sort_values("Date").reset_index(drop=True)).drop(['VNINDEX_volume'], axis=1)
```

Save the dataframe with the international indexes on the Drive.

```python
df_glob.to_csv("df_glob.csv", index=False)
files.download("df_glob.csv")
df_glob.to_parquet("df_glob.parquet", index=False)
files.download("df_glob.parquet")
```

Import the dataframe with the international data to the notebook.

In [9]:
file_path = "/content/drive/My Drive/WQU/10_Capstone_Project/Capstone Project/data/df_glob.parquet"
df_glob = import_data(file_path)

Drive not available or file missing: Error: credential propagation was unsuccessful
Please upload dataframe manually.


Saving df_glob.parquet to df_glob (3).parquet
Loaded df_glob (3).parquet from manual upload


In [10]:
df_glob.head()

,Date,VNINDEX_close,^AXJO_Close,^FTSE_Close,^GSPC_Close,^GSPTSE_Close,^HSI_Close,^KS11_Close,^N225_Close,^STOXX50E_Close,^TWII_Close,CSI300_close
0,2012-01-02,NaN,NaN,NaN,NaN,NaN,NaN,1826.369995,NaN,NaN,NaN,NaN
1,2012-01-03,350.00,4101.200195,5699.899902,1277.060059,12208.400391,18877.410156,1875.410034,NaN,2389.909912,7053.348145,NaN
2,2012-01-04,348.84,4187.799805,5668.500000,1277.300049,12226.500000,18727.310547,1866.219971,8560.110352,2349.889893,7082.937988,2298.753
3,2012-01-05,340.94,4142.700195,5624.299805,1281.060059,12237.400391,18813.410156,1863.739990,8488.709961,2315.750000,7130.827148,2276.385
4,2012-01-06,336.73,4108.500000,5649.700195,1277.810059,12188.599609,18593.060547,1843.140015,8390.349609,2298.649902,7120.477051,2290.601


In [11]:
df_glob.tail()

,Date,VNINDEX_close,^AXJO_Close,^FTSE_Close,^GSPC_Close,^GSPTSE_Close,^HSI_Close,^KS11_Close,^N225_Close,^STOXX50E_Close,^TWII_Close,CSI300_close
3701,2026-03-25,1658.19,8557.599609,10106.799805,6591.899902,32382.599609,25335.949219,5642.209961,53749.621094,5649.330078,33439.109375,4537.466
3702,2026-03-26,1644.63,8525.700195,9972.200195,6477.160156,31887.500000,24856.429688,5460.459961,53603.648438,5565.930176,33337.621094,4477.534
3703,2026-03-27,1672.80,8516.299805,9967.400391,6368.850098,31960.699219,24951.880859,5438.870117,53373.070312,5505.799805,33112.589844,4502.570
3704,2026-03-30,1662.54,8461.000000,10128.000000,6343.720215,31934.900391,24750.789062,5277.299805,51885.851562,5541.790039,32518.160156,4491.950
3705,2026-03-31,1674.49,8481.799805,10176.500000,6528.520020,32768.000000,24788.140625,5052.459961,51063.718750,5569.729980,31722.990234,4450.049


In [12]:
df_glob.shape

(3706, 12)

## 2. Data Cleaning and Exploratory Data Analysis

### 2.1. Domestic Data

### 2.2. International Data